# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.org) library. 

**Dataset summary:**
This dataset contains ordered logistic regression outputs (coefficients, standard errors, log likelihoods, p-values, etc.) across iterations, capturing socio-demographic characteristics, gender roles, adoption of indigenous and modern knowledge, and rangeland management practices among households in Samburu, Isiolo, and Marsabit counties, Northern Kenya.

## Dataset Source
- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)
- [mlcroissant documentation](https://mlcroissant.org/)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
We'll load the dataset metadata and data records using `mlcroissant`. The Croissant schema defines record sets, fields, and their relationships.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (not as dict)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Let's review available record sets, fields, and their `@id` identifiers as defined in the Croissant schema.

Each record set, field, and data column may be referenced by their `@id`. This helps unambiguously refer to data structures for further analysis.

In [ ]:
# List all record sets with their @id and names

print('Available record sets:')
for rs in dataset.record_sets:
    print(f"  - name: {rs.name}\n    @id: {rs.id}")

# For illustration, fetch and show first record (fields, columns) for each record set
for rs in dataset.record_sets:
    print(f"\nRecord set: {rs.name} (@id: {rs.id}) fields:")
    for field in rs.fields:
        print(f"  - name: {field.name}\n    @id: {field.id}")
    # Show up to one record for preview
    try:
        recs = list(dataset.records(record_set=rs.id))
        if recs:
            print('  Example record:')
            print(f'    {recs[0]}')
    except Exception as e:
        print(f"  Error accessing records: {e}")

## 3. Data Extraction
Let's load data from each record set into pandas DataFrames. You can access data using the specific record set `@id` and use field `@id`s for analysis.

In [ ]:
# Extract all record sets to dataframes, keyed by @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record set @ids:")
for rs_id in record_set_ids:
    print(f"  - {rs_id}")

for rs_id in record_set_ids:
    recs = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(recs)
    dataframes[rs_id] = df

# Select the first record set for demonstration
if record_set_ids:
    selected_rs_id = record_set_ids[0]
    print(f"\nColumns in record set '{selected_rs_id}':")
    print(dataframes[selected_rs_id].columns.tolist())
    display(dataframes[selected_rs_id].head())
else:
    print("No record sets found in dataset.")

## 4. Exploratory Data Analysis (EDA)
Let's demonstrate filtering, normalization, and grouping using a numeric field and a grouping field from the chosen record set.

- We select fields by their `@id` for reproducibility and clarity.
- Please adapt the `numeric_field_id` and `group_field_id` to match those present in your dataset record set (use the previous overview to find an appropriate field).

In [ ]:
# Replace these with @id of a real numeric field and grouping field from the dataset
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets:
    if rs.id == selected_rs_id:
        # Find a numeric field (e.g., p-value, estimate, log likelihood, etc.)
        for field in rs.fields:
            if (
                field.data_type in ("schema:Number", "schema:Float", "schema:Integer")
                and numeric_field_id is None
            ):
                numeric_field_id = field.id
            # Find a categorical/grouping field (e.g., variable name)
            if (
                "name" in field.id.lower() or "term" in field.id.lower() or "group" in field.id.lower()
            ) and group_field_id is None:
                group_field_id = field.id
        # If none found, fallback to first column
        if numeric_field_id is None and dataframes[rs.id].shape[1]:
            numeric_field_id = dataframes[rs.id].columns[0]
        if group_field_id is None and dataframes[rs.id].shape[1] > 1:
            group_field_id = dataframes[rs.id].columns[1]
        break

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

df = dataframes[selected_rs_id]
# Remove missing or non-numeric for demonstration
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id]).all() else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalization
if filtered_df[numeric_field_id].notnull().any():
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("No valid numeric values for normalization.")

# Groupby aggregation if group field present
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Let's visualize distributions or a key relationship from the dataset, e.g., plot histogram of the selected numeric field or means by the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} in '{selected_rs_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Barplot for group means if possible
if group_field_id and numeric_field_id and group_field_id in df.columns:
    group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
    plt.figure(figsize=(10,6))
    sns.barplot(x=group_means.values, y=group_means.index, orient='h')
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- This notebook demonstrated:
    - How to load FAIR² datasets defined in the Croissant schema using `mlcroissant`.
    - How to reference record sets and fields by their `@id`.
    - Data extraction, filtering, normalization, group-based statistics, and basic plotting.
- For further analysis, use the field `@id`s from your data dictionary to ensure reproducibility and schema consistency.

For more, see [`mlcroissant`](https://mlcroissant.org/) docs or extend these steps for your specific scientific questions.
